In [37]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [24]:
!pip install torch==2.7.1 tqdm==4.66.4 scikit-learn==1.4.2 pandas numpy

   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/216.1 MB 6.7 MB/s eta 0:00:33
   ---------------------------------------- 1.6/216.1 MB 4.7 MB/s eta 0:00:46
   ---------------------------------------- 2.4/216.1 MB 4.1 MB/s eta 0:00:53
    --------------------------------------- 3.4/216.1 MB 4.4 MB/s eta 0:00:49
    --------------------------------------- 4.2/216.1 MB 4.3 MB/s eta 0:00:49
    --------------------------------------- 5.2/216.1 MB 4.1 MB/s eta 0:00:51
   - -------------------------------------- 5.8/216.1 MB 4.0 MB/s eta 0:00:54
   - -------------------------------------- 6.6/216.1 MB 4.0 MB/s eta 0:00:53
   - -------------------------------------- 7.6/216.1 MB 4.1 MB/s eta 0:00:51
   - -------------------------------------- 8.7/216.1 MB 4.2 MB/s eta 0:00:50
   - -------------------------------------- 9.4/216.1 MB 4.2 MB/s eta 0:00:50
   - -------------------------------------- 9.7/216.1 MB 4.1 MB/s eta 0

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
df=pd.read_csv("diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [6]:
df["Outcome"].value_counts(normalize=True)


Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [8]:
data_diabetes_positive = df.loc[df['Outcome'] == 1]
data_diabetes_positive.info()

<class 'pandas.core.frame.DataFrame'>
Index: 268 entries, 0 to 766
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               268 non-null    int64  
 1   Glucose                   268 non-null    int64  
 2   BloodPressure             268 non-null    int64  
 3   SkinThickness             268 non-null    int64  
 4   Insulin                   268 non-null    int64  
 5   BMI                       268 non-null    float64
 6   DiabetesPedigreeFunction  268 non-null    float64
 7   Age                       268 non-null    int64  
 8   Outcome                   268 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 20.9 KB


In [9]:
data_diabetes_negative = df.loc[df['Outcome'] == 0]
data_diabetes_negative.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 1 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               500 non-null    int64  
 1   Glucose                   500 non-null    int64  
 2   BloodPressure             500 non-null    int64  
 3   SkinThickness             500 non-null    int64  
 4   Insulin                   500 non-null    int64  
 5   BMI                       500 non-null    float64
 6   DiabetesPedigreeFunction  500 non-null    float64
 7   Age                       500 non-null    int64  
 8   Outcome                   500 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 39.1 KB


In [10]:
data_diabetes_negative = data_diabetes_negative.sample(300)
data_diabetes_negative.info() 

<class 'pandas.core.frame.DataFrame'>
Index: 300 entries, 481 to 762
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               300 non-null    int64  
 1   Glucose                   300 non-null    int64  
 2   BloodPressure             300 non-null    int64  
 3   SkinThickness             300 non-null    int64  
 4   Insulin                   300 non-null    int64  
 5   BMI                       300 non-null    float64
 6   DiabetesPedigreeFunction  300 non-null    float64
 7   Age                       300 non-null    int64  
 8   Outcome                   300 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 23.4 KB


In [14]:
data = pd.concat([data_diabetes_negative, data_diabetes_positive])
data = data.sample(frac=1)
data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
360,5,189,64,33,325,31.2,0.583,29,1
458,10,148,84,48,237,37.6,1.001,51,1
480,3,158,70,30,328,35.5,0.344,35,1
604,4,183,0,0,0,28.4,0.212,36,1
448,0,104,64,37,64,33.6,0.510,22,1
...,...,...,...,...,...,...,...,...,...
732,2,174,88,37,120,44.5,0.646,24,1
253,0,86,68,32,0,35.8,0.238,25,0
633,1,128,82,17,183,27.5,0.115,22,0
578,10,133,68,0,0,27.0,0.245,36,0


In [28]:
X = data.drop(columns=['Outcome']).values.astype('float32') 
y = data['Outcome'].values.astype('float32') 

X.shape, y.shape

((568, 8), (568,))

In [29]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2)

X_train.shape, X_val.shape

((454, 8), (114, 8))

In [30]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train[:3]

array([[ 0.58218694,  1.1412001 , -0.36410072, -1.2825079 , -0.71670264,
        -1.0529629 , -0.9254588 ,  1.3742498 ],
       [ 2.0505917 ,  0.30067673, -3.447346  , -1.2825079 , -0.71670264,
         2.5267503 ,  0.2948674 ,  0.52365756],
       [-1.1798989 , -0.01062825, -0.06572215, -1.2825079 , -0.71670264,
        -1.001824  , -0.840036  , -1.0924678 ]], dtype=float32)

In [31]:
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = DiabetesDataset(X_train, y_train)
val_ds = DiabetesDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

len(train_ds), len(val_ds)

(454, 114)

In [32]:
class DiabetesNet(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),  # one logit
        )

    def forward(self, x):
        # Output shape: (batch_size,)
        return self.net(x).squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DiabetesNet(in_features=X_train.shape[1]).to(device)
model

Using device: cpu


DiabetesNet(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [33]:
pos_weight_value = 268/300
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [34]:
def train_one_epoch(epoch_idx: int):
    model.train()
    losses = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch_idx+1}", leave=False)

    for X_batch, y_batch in progress_bar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return float(np.mean(losses))


def evaluate():
    model.eval()
    all_probs = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            all_probs.append(probs)
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_probs = np.concatenate(all_probs).reshape(-1)
    all_preds = np.concatenate(all_preds).reshape(-1)
    all_targets = np.concatenate(all_targets).reshape(-1)

    acc = accuracy_score(all_targets, all_preds)
    return acc, all_targets, all_preds

In [35]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)
    val_acc, _, _ = evaluate()
    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train_loss = {train_loss:.4f} | val_acc = {val_acc:.4f}"
    )

print("Training finished!")

Epoch 01/20 | train_loss = 0.6492 | val_acc = 0.6842


Epoch 02/20 | train_loss = 0.6301 | val_acc = 0.6930


Epoch 03/20 | train_loss = 0.6113 | val_acc = 0.7018


Epoch 04/20 | train_loss = 0.5977 | val_acc = 0.7368


Epoch 05/20 | train_loss = 0.5868 | val_acc = 0.7544


...

Epoch 16/20 | train_loss = 0.4482 | val_acc = 0.8158


Epoch 17/20 | train_loss = 0.4628 | val_acc = 0.8070


Epoch 18/20 | train_loss = 0.4546 | val_acc = 0.8158


Epoch 19/20 | train_loss = 0.4431 | val_acc = 0.8246


Epoch 20/20 | train_loss = 0.4287 | val_acc = 0.8158
Training finished!


In [38]:
val_acc, y_true, y_pred = evaluate()
print("Validation accuracy:", val_acc)
print()
print("Classification report:")
print(classification_report(y_true, y_pred, digits=3))

Validation accuracy: 0.8157894736842105

Classification report:
              precision    recall  f1-score   support

         0.0      0.800     0.909     0.851        66
         1.0      0.846     0.688     0.759        48

    accuracy                          0.816       114
   macro avg      0.823     0.798     0.805       114
weighted avg      0.819     0.816     0.812       114



In [40]:
class DiabetesNet2(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)
        
model = DiabetesNet2(in_features=X_train.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



In [41]:
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)
    val_acc, _, _ = evaluate()
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")


Epoch 1/20 | train_loss=0.6790 | val_acc=0.6491


Epoch 2/20 | train_loss=0.6581 | val_acc=0.7544


Epoch 3/20 | train_loss=0.6166 | val_acc=0.7544


Epoch 4/20 | train_loss=0.5923 | val_acc=0.7982


Epoch 5/20 | train_loss=0.5481 | val_acc=0.8070


...

Epoch 16/20 | train_loss=0.4708 | val_acc=0.8509


Epoch 17/20 | train_loss=0.5105 | val_acc=0.8246


Epoch 18/20 | train_loss=0.4673 | val_acc=0.8070


Epoch 19/20 | train_loss=0.4626 | val_acc=0.7982


Epoch 20/20 | train_loss=0.4372 | val_acc=0.8158


In [42]:
val_acc, y_true, y_pred = evaluate()
print("Validation accuracy:", val_acc)
print()
print("Classification report:")
print(classification_report(y_true, y_pred, digits=3))

Validation accuracy: 0.8157894736842105

Classification report:
              precision    recall  f1-score   support

         0.0      0.817     0.879     0.847        66
         1.0      0.814     0.729     0.769        48

    accuracy                          0.816       114
   macro avg      0.815     0.804     0.808       114
weighted avg      0.816     0.816     0.814       114

